# Model comparison: RandomForest and HistGradientBoosting

`01-train-harness.ipynb` already covers the shared 5-fold CV + final-test
evaluation with LogisticRegression across all feature sets. This notebook
covers the other model types from the original plan (Random Forest,
LightGBM-or-fallback) and compares them against that LogReg baseline on
the same frozen split.

Run on real data via `src/data.py`'s `load_dataset()` -- the same frozen
`data/split.csv` join everything else in this repo uses, so these numbers
are directly comparable to the LogReg ones.

**Memory note**: this was run on a shared desktop machine with ~1.5-1.8GB
of RAM actually available (not a dedicated box), which matters for what's
included below -- see the HistGBM section.

> **Note**: the comparison section below reads `results/results_rf_hgb.csv`, which lands in this repo via PR #13 (`tree-model-sweep`). If that file isn't present yet when you run this, merge that PR first or re-run the RandomForest/HistGBM cells above to regenerate it.

In [1]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), "..", "src"))

import config as C
from data import load_dataset
from binning import BandSpec
from pipeline import build_pipeline
from metrics import evaluate, log_result

print(C.describe())
splits = load_dataset()
for name, d in splits.items():
    print(f"{name:>5}: {d.shape}")

seed=42 bands=4(fixed) years=2019-2023 window=2y split=data/split.csv kaggle=False
train: (30797, 10)
  val: (6600, 10)
 test: (6600, 10)


## RandomForest, all 5 feature sets

Sparse-native (no dense conversion needed), so this is the safe one to run
on all feature sets including the wide TF-IDF ones. `n_estimators=300`,
`class_weight="balanced_subsample"` (see `models.py`). `n_jobs` capped at 2
rather than the default `-1` to keep memory pressure down on this machine.

In [2]:
OUT = "../results/results_rf_hgb.csv"  # separate file -- results.csv is not touched here

spec = BandSpec("fixed", [0.0, 2.0, 7.0, float("inf")],
                ["Uncited", "Low", "Medium", "High"])
label_of = dict(enumerate(spec.labels))
train, val = splits["train"], splits["val"]
ytr = train["band"].map(label_of)
yva = val["band"].map(label_of)

configs = [
    ("B1_title_only", "title only"),
    ("A3_venue_only", "venue only"),
    ("B3_meta_only", "metadata only (no venue)"),
    ("A2_no_venue", "title + metadata"),
    ("A1_all", "everything (with venue)"),
]
for fset, label in configs:
    pipe = build_pipeline(feature_set=fset, model_name="random_forest",
                          model_kwargs={"n_jobs": 2})
    pipe.fit(train, ytr)
    pred = pipe.predict(val)
    r = evaluate(yva, pred, labels=spec.labels, y_train=ytr,
                 model_name=f"RandomForest / {label}", feature_set=fset,
                 eval_split="val", verbose=False)
    log_result(r, notes="random_forest, n_estimators=300, class_weight=balanced_subsample", path=OUT)
    print(f"RF {label:30s} acc={r['accuracy']:.3f} macroF1={r['macro_f1']:.3f}")

RF title only                     acc=0.375 macroF1=0.359
RF venue only                     acc=0.415 macroF1=0.381
RF metadata only (no venue)       acc=0.410 macroF1=0.403
RF title + metadata               acc=0.409 macroF1=0.388
RF everything (with venue)        acc=0.413 macroF1=0.389


## HistGradientBoosting -- narrow feature sets only

LightGBM isn't installed (`requirements.txt`), so `get_model("lightgbm")`
falls back to sklearn's `HistGradientBoostingClassifier`, which requires
**dense** input. Densifying the wide TF-IDF feature sets (title-inclusive,
~1600 columns x 30,797 rows) OOM'd on this machine (exit 137) -- so only
the two narrow, already-cheap-to-densify feature sets are run here. The
wide ones are left to RandomForest above, which stays sparse-native.

In [3]:
for fset, label in [("A3_venue_only", "venue only"),
                   ("B3_meta_only", "metadata only (no venue)")]:
    pipe = build_pipeline(feature_set=fset, model_name="hist_gbm")
    pipe.fit(train, ytr)
    pred = pipe.predict(val)
    r = evaluate(yva, pred, labels=spec.labels, y_train=ytr,
                 model_name=f"HistGBM / {label}", feature_set=fset,
                 eval_split="val", verbose=False)
    log_result(r, notes="hist_gbm (sklearn fallback), narrow feature set only (memory-safe to densify)", path=OUT)
    print(f"HGB {label:30s} acc={r['accuracy']:.3f} macroF1={r['macro_f1']:.3f}")

HGB venue only                     acc=0.366 macroF1=0.317
HGB metadata only (no venue)       acc=0.438 macroF1=0.422


## Comparison against the LogReg baseline

`results.csv` holds the collaborator's LogReg CV + final-test numbers
(from `01-train-harness.ipynb`); `results_rf_hgb.csv` holds the RF/HistGBM
numbers from above. Same frozen split, same feature sets, same metric --
directly comparable.

In [4]:
import pandas as pd

main = pd.read_csv("../results/results.csv")
rf = pd.read_csv("../results/results_rf_hgb.csv")

label_map = {
    "B1_title_only": "title only", "A3_venue_only": "venue only",
    "B3_meta_only": "metadata only (no venue)", "A2_no_venue": "title + metadata",
    "A1_all": "everything (with venue)",
}
rows = []
for fs, label in label_map.items():
    cv = main[(main.feature_set == fs) & (main.eval_split == "cv")]
    test = main[(main.feature_set == fs) & (main.eval_split == "test")]
    rf_row = rf[(rf.feature_set == fs) & (rf.model.str.startswith("RandomForest"))]
    hgb_row = rf[(rf.feature_set == fs) & (rf.model.str.startswith("HistGBM"))]
    rows.append({
        "feature_set": label,
        "LogReg (CV)": cv.macro_f1.values[0] if len(cv) else None,
        "LogReg (test)": test.macro_f1.values[0] if len(test) else None,
        "RandomForest (val)": rf_row.macro_f1.values[0] if len(rf_row) else None,
        "HistGBM (val)": hgb_row.macro_f1.values[0] if len(hgb_row) else None,
    })

comparison = pd.DataFrame(rows).set_index("feature_set")
comparison

                          LogReg (CV)  LogReg (test)  RandomForest (val)  HistGBM (val)
feature_set
title only                     0.3715         0.3612              0.3592            NaN
venue only                     0.3851         0.3898              0.3814         0.3172
metadata only (no venue)       0.4266         0.4229              0.4025         0.4224
title + metadata               0.4243         0.4197              0.3875            NaN
everything (with venue)        0.4481         0.4498              0.3891            NaN

## Conclusion

LogReg beats or ties both tree models on every feature set except
metadata-only, where HistGBM ties it (0.422 vs 0.423 CV). On the feature
set that matters most -- **everything, incl. venue** -- LogReg wins
clearly: **0.450 vs RandomForest's 0.389 macro-F1**, a 6-point gap.

Reading: on this problem, the dominant features are sparse TF-IDF (title)
and one-hot categoricals (venue, subfield, topic) -- exactly the regime
where a well-regularised linear model tends to beat tree ensembles, which
split on one feature at a time and don't exploit the sparse structure as
efficiently. This matches the plan's own expectation in `models.py`'s
docstring and is why LogReg is the model used in the demo app and the
final comparison table, not a tree ensemble.

MLP was not run -- lowest priority on the plan's own cut list, and not
worth the time against this machine's memory constraints this close to
the deadline.